# Compound Heatwave Event

In this notebook, we'll be identifying days when three major heatwave conditions occur simultaneously across the SCE service territory and California, which represents a severe spatially compounding event that challenges both power demand and distribution infrastructure.

### Real-World Motivation: July 6–8, 2018 Heatwave

A multi-day heatwave impacted Southern California in early July 2018. On July 7th, approximately
**34,000 customers** of the Los Angeles Department of Water and Power experienced power outages,
some lasting up to 24 hours. An offshore flow pattern pushed back the normal moderating influence
of the Pacific Ocean, producing startling early-morning temperatures. Many Southern California
locations reported temperatures **above 80°F before dawn**.

This event exemplifies the spatially compound heatwave stress scenario analyzed in this notebook:

| Sub-condition | Indicator | Threshold |
|---|---|---|
| CA statewide heat | Daily max temp across California (d01 45 km) | > 90°F in ≥ 50% of CA land grid cells, 1 day |
| SCE multi-day heat | Daily max temp in SCE Service Territory (d03 3 km) | > 90°F for 3 consecutive days in ≥ 50% of SCE grid cells |
| SCE nighttime heat | Daily min temp in SCE Service Territory (d03 3 km) | > 65°F for 3 consecutive days in ≥ 50% of SCE grid cells |

**Intended Application:** As a user, I want to **<span style="color:red">stress-test grid resiliency by quantifying how often three simultaneous heatwave conditions challenge both energy demand and supply within the SCE service territory, and how that frequency changes between GWL 0.8°C and 2.0°C.</span>**

**Runtime:** ~20 min. to run through the whole notebook with pre-cached CA statewide conditions. Modifications to different queries may increase/decrease the overall runtime.

#### Imports

In [ ]:
import os
import sys

import climakitae as ck
import dask
import numpy as np
import pandas as pd
import xarray as xr
from climakitae.core.data_export import export
from dask.diagnostics import ProgressBar

sys.path.append(os.path.abspath('.'))
from wg10_helpers import (
    plot_cdd_8760,
    plot_gwl_bar_chart,
)

cd = ck.ClimateData(verbosity=-1)

## User Selections

Modify these parameters to change the compound event definition, spatial thresholds, or data resolution.

In [ ]:
GWLS = [0.8, 2.0]
territory = "Southern California Edison"  # SCE service territory
ca_territory = "CA"  # California statewide
resolution = "d03"  # 3 km for SCE domain
ca_resolution = "d01"  # 45 km for CA statewide

warming_level_window = 15  # ±window yr around GWL crossing (15 → 30 yr total)
n_years = warming_level_window * 2.0  # = 30.0

# Fixed temperature thresholds (all conditions)
tmax_threshold_f = 90.0  # max temp threshold (°F)
tmin_threshold_f = 65.0  # min temp threshold (°F)
consec_days = 3  # consecutive days for SCE conditions

# Spatial coverage required for each sub-condition
spatial_threshold = 0.5

# GWL display colors and labels for bar charts
gwl_colors = {0.8: "#4472C4", 2.0: "#C00000"}
gwl_labels = {0.8: "GWL 0.8°C", 2.0: "GWL 2.0°C"}

## Retrieve Data

In [ ]:
procs = {
    "warming_level": {
        "warming_levels": GWLS,
        "add_dummy_time": False,
        "warming_level_window": warming_level_window,
    },
    "clip": territory,
    "filter_unadjusted_models": "yes",
    "convert_units": "degF",
}

# SCE t2max pull (3 km d03)
t2max_sce = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label(resolution)
    .variable("t2max")
    .processes(procs)
    .get()
)
print("SCE t2max dims:", dict(t2max_sce.dims))

In [ ]:
# CA statewide boolean conditions are loaded from cache rather than re-computed.
# The raw d01 pull + spatial fraction computation takes ~1 hr, so results were
# pre-computed and saved as NetCDFs (one per GWL) containing a (sim, time_delta)
# boolean DataArray: True on days when tmax > threshold in ≥50% of CA land cells.
ca_heat_cache = {}

for gwl in GWLS:
    gwl_tag = f"gwl{str(gwl).replace('.', '')}"
    cache_path = f"data/ca_heat_cond_{gwl_tag}.nc"
    print(f"GWL {gwl}: loading CA condition from cache → {cache_path}")
    ca_heat_cache[gwl] = xr.open_dataarray(cache_path)

In [ ]:
# SCE t2min pull (3 km d03)
procs["clip"] = territory
t2min_sce = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label(resolution)
    .variable("t2min")
    .processes(procs)
    .get()
)
print("SCE t2min dims:", dict(t2min_sce.dims))

## Compute Spatial Mask

Derives a boolean mask of valid SCE land cells from the first time step of temperature data.

In [ ]:
# Grabbing a spatial mask — True = valid land cell
sce_mask = (
    ~t2max_sce["t2max"].isel(warming_level=0, time_delta=0, sim=0).isnull()
).compute()

## Count Stress-Test Events Per GWL

For each GWL and simulation, `count_stress_events` evaluates the three sub-conditions daily:
- **CA heat**: fraction of valid CA grid cells with tmax > threshold ≥ spatial_threshold (pre-computed)
- **SCE heat**: fraction of valid SCE cells meeting a `consec_days`-day rolling tmax threshold
- **SCE nighttime**: same rolling approach applied to tmin

All three must be `True` on the same day to count as a compound event.

In [ ]:
def count_stress_events(
    t2max_sce_da,
    t2min_sce_da,
    ca_heat_precomputed,
    sce_mask,
    tmax_thresh=90.0,
    tmin_thresh=65.0,
    consec=3,
    spatial_thresh=0.5,
    n_years=30.0,
):
    """
    Compute compound heat stress event frequency for a single GWL.

    A compound event day requires all three conditions simultaneously:
      1. CA statewide  : tmax > tmax_thresh in ≥ spatial_thresh of CA grids (pre-computed boolean)
      2. SCE territory : tmax > tmax_thresh for `consec` consecutive days in ≥ spatial_thresh of SCE grids
      3. SCE territory : tmin > tmin_thresh for `consec` consecutive days in ≥ spatial_thresh of SCE grids

    The SCE spatial fraction is computed only over valid land cells (non-null in sce_mask) to
    avoid diluting the fraction with ocean/boundary cells.

    Parameters
    ----------
    t2max_sce_da : xr.DataArray
        Daily max temperature over SCE territory (°F), dims (sim, time_delta, y, x).
    t2min_sce_da : xr.DataArray
        Daily min temperature over SCE territory (°F), dims (sim, time_delta, y, x).
    ca_heat_precomputed : xr.DataArray
        Pre-cached boolean CA condition, dims (sim, time_delta). True on days the CA
        statewide fraction threshold is met.
    sce_mask : xr.DataArray
        Boolean (y, x) mask; True = valid SCE land cell.
    tmax_thresh : float
        Hot-day threshold in °F. Default 90.
    tmin_thresh : float
        Warm-night threshold in °F. Default 65.
    consec : int
        Number of consecutive days required for SCE conditions. Default 3.
    spatial_thresh : float
        Minimum fraction of valid SCE cells that must meet the threshold. Default 0.5.
    n_years : float
        Sample window length in years; converts day counts to annual rates.

    Returns
    -------
    dict of lazy xr.DataArrays:
        compound         : (sim,)              compound events / yr
        compound_bool    : (sim, time_delta)   True on compound event days
        sce_heat_spatial : (sim, time_delta)   True when SCE tmax condition met
        sce_tmin_spatial : (sim, time_delta)   True when SCE tmin condition met
    """
    # Flatten to 1D cell index; keep only valid land cells for spatial fraction
    valid_cells = sce_mask.stack(cell=["y", "x"]).values  # boolean 1D array

    def spatial_consec_flag(da, thresh):
        """True on day N if thresh is exceeded for `consec` consecutive days in ≥ spatial_thresh of valid cells."""
        consec_hit = (da > thresh).rolling(
            time_delta=consec, min_periods=consec
        ).sum() >= consec
        stacked = consec_hit.stack(cell=["y", "x"])
        return stacked.isel(cell=valid_cells).mean(dim="cell") >= spatial_thresh

    sce_heat_spatial = spatial_consec_flag(t2max_sce_da, tmax_thresh)
    sce_tmin_spatial = spatial_consec_flag(t2min_sce_da, tmin_thresh)

    compound = ca_heat_precomputed & sce_heat_spatial & sce_tmin_spatial

    return {
        "compound": compound.sum(dim="time_delta") / n_years,
        "compound_bool": compound,
        "sce_heat_spatial": sce_heat_spatial,
        "sce_tmin_spatial": sce_tmin_spatial,
    }

#### Here, we will count the number of compound events by GWL, compute it lazily, and present the progress using Dask ProgressBars.

In [ ]:
records = []
date_records = []
sce_heat_spatial_by_gwl = {}
sce_tmin_spatial_by_gwl = {}

for gwl in GWLS:
    lazy = count_stress_events(
        t2max_sce.sel(warming_level=gwl)["t2max"],
        t2min_sce.sel(warming_level=gwl)["t2min"],
        ca_heat_precomputed=ca_heat_cache[gwl],
        sce_mask=sce_mask,
        tmax_thresh=tmax_threshold_f,
        tmin_thresh=tmin_threshold_f,
        consec=consec_days,
        spatial_thresh=spatial_threshold,
        n_years=n_years,
    )

    # Compute each GWL separately so the ProgressBar gives per-GWL feedback.
    print(f"Computing GWL {gwl}°C...")
    with ProgressBar():
        result = dict(zip(lazy.keys(), dask.compute(*lazy.values())))

    scalar_per_sim = result["compound"]  # events/yr per sim (scalar)
    compound_bool_ts = result["compound_bool"]  # (sim, time_delta) boolean
    sce_heat_spatial_ts = result["sce_heat_spatial"]
    sce_tmin_spatial_ts = result["sce_tmin_spatial"]

    # Store sub-condition timeseries for export
    sce_heat_spatial_by_gwl[gwl] = sce_heat_spatial_ts
    sce_tmin_spatial_by_gwl[gwl] = sce_tmin_spatial_ts

    # Record every compound event day index (used downstream to select representative cell/year)
    for sim_name in compound_bool_ts.sim.values:
        hit_idxs = np.where(compound_bool_ts.sel(sim=sim_name).values)[0]
        for t_idx in hit_idxs:
            date_records.append(
                {"gwl": gwl, "sim": sim_name, "time_delta_idx": int(t_idx)}
            )

    print(
        f"GWL {gwl}°C → median {float(scalar_per_sim.median()):.4f} events/yr ({len(scalar_per_sim)} sims)"
    )

    for sim_name, val in zip(scalar_per_sim.sim.values, scalar_per_sim.values):
        records.append(
            {
                "gwl": gwl,
                "sim": sim_name,
                "events_per_year": float(val),
                "territory": "SCE",
            }
        )

results_df = pd.DataFrame(records)
dates_df = pd.DataFrame(date_records)
print(f"\nCompound event dates: {len(dates_df)} total hit days across all sims/GWLs")

## Save Outputs

Saves four sets of outputs:
- `stress_test_results.csv` — per-sim annual compound event frequency by GWL
- `compound_event_dates.csv` — (gwl, sim, time_delta_idx) for every compound event hit
- `ca_heat_spatial_gwl*.nc`, `sce_heat_spatial_gwl*.nc`, `sce_tmin_spatial_gwl*.nc` — per-GWL
  boolean timeseries of each sub-condition meeting its spatial threshold

In [ ]:
results_df.to_csv("data/stress_test_results.csv", index=False)
dates_df.to_csv("data/compound_event_dates.csv", index=False)
print(f"Saved → data/stress_test_results.csv")
print(f"Saved → data/compound_event_dates.csv")
print(dates_df.groupby("gwl").agg(n_hits=("time_delta_idx", "count")).to_string())

In [ ]:
# Save sub-condition timeseries per GWL (used for sub-condition bar charts)
for gwl in GWLS:
    gwl_tag = f"gwl{str(gwl).replace('.', '')}"
    for name, da in [
        ("ca_heat_spatial", ca_heat_cache[gwl]),
        ("sce_heat_spatial", sce_heat_spatial_by_gwl[gwl]),
        ("sce_tmin_spatial", sce_tmin_spatial_by_gwl[gwl]),
    ]:
        da.attrs = {}
        export(da, f"data/{name}_{gwl_tag}")
print("Done saving sub-condition timeseries.")

## 8760 Analysis

Now, we are going to pull out an 8760 of temperature for each GWL. The workflow:
1. **Cell selection**: find an SCE land cell where both `tmax` and `tmin` locally exceed their thresholds
   on at least one domain-level compound event day.
2. **Year selection**: for each GWL, find the calendar year with the most compound event hits for
   the selected simulation, using `centered_year` from the cached NetCDF to convert `time_delta`
   integer offsets to actual calendar years.
3. **Hourly pull**: fetch hourly `t2` for that cell and year from the 1-hr WRF catalog.
4. **Export**: concatenate across GWLs and save as `t2_8760_overlay.nc`.

In [ ]:
# Read back saved results for 8760 analysis and figure generation
results_df = pd.read_csv("data/stress_test_results.csv")
dates_df = pd.read_csv("data/compound_event_dates.csv")

In [ ]:
# Find a representative SCE land cell that locally satisfies both tmax > threshold
# AND tmin > threshold on at least one domain-level compound event day.
# This cell is used for the hourly 8760 temperature pull below.

# Selecting one of the simulations, you can change this to any of the other 4 other simulations
SIM_SEL = "WRF_UCLA_TaiESM1_ssp370_day_d03_r1i1p1f1"
GWL_SEL = GWLS[
    -1
]  # Use highest GWL — more compound events → higher chance of finding a valid cell

compound_idxs = dates_df[(dates_df["gwl"] == GWL_SEL) & (dates_df["sim"] == SIM_SEL)][
    "time_delta_idx"
].values

if len(compound_idxs) == 0:
    # Fall back to the sim with the most hits at this GWL
    SIM_SEL = dates_df[dates_df["gwl"] == GWL_SEL]["sim"].value_counts().idxmax()
    compound_idxs = dates_df[
        (dates_df["gwl"] == GWL_SEL) & (dates_df["sim"] == SIM_SEL)
    ]["time_delta_idx"].values

with ProgressBar():
    print(
        "Loading tmax for selected hits and finding which gridcell has the most hits..."
    )
    tmax_on_hits = (
        t2max_sce.sel(warming_level=GWL_SEL)["t2max"]
        .sel(sim=SIM_SEL)
        .isel(time_delta=compound_idxs)
        .compute()
    )
    print(
        "Loading tmin for selected hits and finding which gridcell has the most hits..."
    )
    tmin_on_hits = (
        t2min_sce.sel(warming_level=GWL_SEL)["t2min"]
        .sel(sim=SIM_SEL)
        .isel(time_delta=compound_idxs)
        .compute()
    )

# Cells where both local temperature conditions are met on at least one compound event day
both_hit = ((tmax_on_hits > tmax_threshold_f) & (tmin_on_hits > tmin_threshold_f)).any(
    dim="time_delta"
)
both_hit_land = both_hit & sce_mask  # restrict to valid land cells

candidates = np.argwhere(both_hit_land.values)
y_cell, x_cell = int(candidates[0][0]), int(candidates[0][1])

cell_lat = float(both_hit_land.isel(y=y_cell, x=x_cell).lat.item())
cell_lon = float(both_hit_land.isel(y=y_cell, x=x_cell).lon.item())
print(
    f"Selected cell: lat={cell_lat:.4f}, lon={cell_lon:.4f}  ({len(candidates)} candidates in SCE territory, sim={SIM_SEL})"
)

In [ ]:
# For each GWL, identify the calendar year with the highest number of compound event days
# for the selected simulation. This year is then used for the hourly 8760 pull.
#
# `centered_year` is stored in the cached sub-condition NetCDF and represents the
# approximate center of the 30-year warming-level window (e.g., 2045 for GWL 2.0).
# time_delta is an integer day offset from the start of that window, so:
#   calendar_year ≈ centered_year + time_delta / 365.25

year_info = {}

for gwl in GWLS:
    gwl_tag = f"gwl{str(gwl).replace('.', '')}"
    ca_bool = xr.open_dataarray(f"data/ca_heat_spatial_{gwl_tag}.nc")
    td_coord = ca_bool.coords["time_delta"].values
    cy = float(ca_bool.sel(sim=SIM_SEL).coords["centered_year"].values)

    day_idxs = dates_df[(dates_df["gwl"] == gwl) & (dates_df["sim"] == SIM_SEL)][
        "time_delta_idx"
    ].values

    if len(day_idxs) == 0:
        print(f"GWL {gwl}: no compound events for sim {SIM_SEL} — skipping")
        continue

    td_vals = td_coord[day_idxs]
    cal_years = (cy + td_vals / 365.25).astype(int)
    year_counts = pd.Series(cal_years).value_counts()
    target_year = int(year_counts.index[0])  # year with the most compound hits

    # Compute time_delta slice bounds for this target year within the 30-yr window
    year_offset = target_year - int(cy)
    td_start = year_offset * 365
    td_end = td_start + 365

    year_info[gwl] = {
        "target_year": target_year,
        "centered_year": cy,
        "td_start": td_start,
        "td_end": td_end,
        "n_hits": int(year_counts.iloc[0]),
    }
    print(
        f"GWL {gwl}°C → target year {target_year} ({year_counts.iloc[0]} compound hits in this year, centered_year={cy:.0f})"
    )

In [ ]:
# Pull hourly t2 for the selected cell and target year for each GWL.
# Clipping to a single (lat, lon) point keeps the pull fast (~8760 hourly values per year).
# Pulls use source_id="TaiESM1" to match SIM_SEL, and table_id="1hr" for hourly data.

slices = []

for gwl in GWLS:
    if gwl not in year_info:
        print(f"GWL {gwl}: skipped (no compound events)")
        continue

    target_year = year_info[gwl]["target_year"]
    print(f"GWL {gwl}°C — pulling hourly t2 for {target_year} ...")

    cd.reset()
    t2_yr = (
        cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .source_id("TaiESM1")  # Keeping the same simulation as `SIM_SEL`
        .table_id("1hr")
        .grid_label(resolution)
        .variable("t2")
        .processes(
            {
                "time_slice": (target_year, target_year),
                "filter_unadjusted_models": "yes",
                "convert_units": "degF",
                "clip": (
                    (cell_lat, cell_lon)
                ),  # single-point clip → returns nearest grid cell
            }
        )
        .get()
    )

    n_hrs = len(t2_yr.time)
    # Replace calendar timestamps with integer hour index (0–8759) for alignment across GWLs
    da = (
        t2_yr.assign_coords(hour=("time", range(n_hrs)))
        .swap_dims({"time": "hour"})
        .drop_vars("time")
        .expand_dims(warming_level=[gwl])
    )
    da.attrs["target_year"] = target_year
    da.attrs["centered_year"] = year_info[gwl]["centered_year"]
    da.attrs["sim"] = SIM_SEL
    slices.append(da)

t2_8760 = xr.concat(slices, dim="warming_level")
export(t2_8760, "data/t2_8760_overlay")
print("t2_8760 dims:", dict(t2_8760.dims))

**Now that we've computed compound stress-test event frequency across the SCE service territory for GWLs 0.8 and 2.0, we can move on below to visualizing our results.**

---

# Figures

In [ ]:
# Figure 1: CA statewide heat condition frequency bar chart
ca_vals = {
    gwl_labels[g]: float(
        xr.open_dataarray(
            f"data/ca_heat_spatial_gwl{str(g).replace('.', '')}.nc"
        )
        .sum(dim="time_delta")
        .mean()
        .values
    )
    / n_years
    for g in GWLS
}

plot_gwl_bar_chart(
    bars_by_label=ca_vals,
    title=f"CA Statewide Heat Condition Frequency\n(tmax > {tmax_threshold_f:.0f}°F in ≥50% of CA grids)",
    ylabel="Days per year",
    out_path="figures/ca_heat_frequency_bar_chart.png",
    colors=[gwl_colors[g] for g in GWLS],
    figsize=(5, 5),
)

In [ ]:
# Figure 2: SCE territory heat condition frequency bar chart
sce_heat_vals = {
    gwl_labels[g]: float(
        xr.open_dataarray(
            f"data/sce_heat_spatial_gwl{str(g).replace('.', '')}.nc"
        )
        .sum(dim="time_delta")
        .mean()
        .values
    )
    / n_years
    for g in GWLS
}

plot_gwl_bar_chart(
    bars_by_label=sce_heat_vals,
    title=(
        f"SCE Territory Heat Condition Frequency\n"
        f"(tmax > {tmax_threshold_f:.0f}°F, {consec_days} consecutive days, in ≥{spatial_threshold:.0%} of SCE grids)"
    ),
    ylabel="Days per year",
    out_path="figures/sce_heat_frequency_bar_chart.png",
    colors=[gwl_colors[g] for g in GWLS],
    figsize=(5, 5),
)

In [ ]:
# Figure 3: SCE territory nighttime heat condition frequency bar chart
sce_tmin_vals = {
    gwl_labels[g]: float(
        xr.open_dataarray(
            f"data/sce_tmin_spatial_gwl{str(g).replace('.', '')}.nc"
        )
        .sum(dim="time_delta")
        .mean()
        .values
    )
    / n_years
    for g in GWLS
}

plot_gwl_bar_chart(
    bars_by_label=sce_tmin_vals,
    title=(
        f"SCE Territory Nighttime Heat Condition Frequency\n"
        f"(tmin > {tmin_threshold_f:.0f}°F, {consec_days} consecutive days, in ≥{spatial_threshold:.0%} of SCE grids)"
    ),
    ylabel="Days per year",
    out_path="figures/sce_tmin_frequency_bar_chart.png",
    colors=[gwl_colors[g] for g in GWLS],
    figsize=(5, 5),
)

In [ ]:
# Figure 4: Compound heatwave event frequency bar chart
median_df = results_df.groupby("gwl")["events_per_year"].median().reset_index()

plot_gwl_bar_chart(
    bars_by_label={
        gwl_labels[g]: float(
            median_df.loc[median_df["gwl"] == g, "events_per_year"].iloc[0]
        )
        for g in GWLS
    },
    title=(
        f"Compound Heatwave Event Frequency\n"
        f"(tmax>{tmax_threshold_f:.0f}°F 3-day SCE + tmin>{tmin_threshold_f:.0f}°F 3-day SCE + tmax>{tmax_threshold_f:.0f}°F 1-day CA)"
    ),
    ylabel="Compound events per year",
    out_path="figures/compound_heatwave_bar_chart.png",
    colors=[gwl_colors[g] for g in GWLS],
    figsize=(5, 5),
)

In [ ]:
# Figure 5: CDD (cooling degree-day) area plot for the selected cell's 8760 hourly profile
# Computed separately per GWL, each saved as its own PNG.
da_8760 = xr.open_dataarray("data/t2_8760_overlay.nc").isel(sim=0)
for gwl in GWLS:
    gwl_tag = f"gwl{str(gwl).replace('.', '')}"
    plot_cdd_8760(
        da_8760.sel(warming_level=gwl),
        base_temp_f=65.0,
        gwl=gwl,
        # Same color for both GWLs — the metric is identical, only the GWL slice differs
        gwl_colors={0.8: "#4472C4", 2.0: "#4472C4"},
        out_path=f"figures/cdd_8760_{gwl_tag}.png",
    )